# Malaria cell image classification

This notebook trains and evaluates an example binary CNN. The image dataset is not included in this repository; configure `MALARIA_DATA_DIR` as described in `README.md` before running. The held-out test split is evaluated only after training.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

tf.keras.utils.set_random_seed(42)

DATASET_ROOT = Path(os.environ.get("MALARIA_DATA_DIR", "data/Malaria Cells")).expanduser()
TRAIN_DIR = DATASET_ROOT / "training_set"
TEST_DIR = DATASET_ROOT / "testing_set"
if not TRAIN_DIR.is_dir() or not TEST_DIR.is_dir():
    raise FileNotFoundError(
        "Expected training_set/ and testing_set/ under "
        f"{DATASET_ROOT.resolve()}. See README.md for the required layout."
    )

class_names = sorted(path.name for path in TRAIN_DIR.iterdir() if path.is_dir())
test_class_names = sorted(path.name for path in TEST_DIR.iterdir() if path.is_dir())
if len(class_names) != 2 or test_class_names != class_names:
    raise ValueError(
        "Training and testing splits must contain the same two class folders; "
        f"found training={class_names}, testing={test_class_names}."
    )


In [ ]:
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

training_set = train_datagen.flow_from_directory(
    TRAIN_DIR,
    classes=class_names,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    seed=42,
)
test_set = test_datagen.flow_from_directory(
    TEST_DIR,
    classes=class_names,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

class_indices = training_set.class_indices
print("Class indices:", class_indices)


In [ ]:
model = Sequential([
    Input(shape=(*IMAGE_SIZE, 3)),
    Conv2D(64, kernel_size=3, activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, kernel_size=3, activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()


In [ ]:
# Epoch and step counts are derived from the available images by Keras.
history = model.fit(training_set, epochs=10)


In [ ]:
plt.plot(history.history["loss"], label="training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
# Evaluate the held-out test split once after training. Use a separate
# validation split for model selection or hyperparameter tuning.
test_metrics = model.evaluate(test_set, return_dict=True)
test_metrics


## Optional single-image prediction

Set `MALARIA_IMAGE_FILE` to an image path before starting Jupyter to run this example. Class labels are resolved from the directory indices rather than hard-coded.


In [ ]:
image_path_value = os.environ.get("MALARIA_IMAGE_FILE")
if not image_path_value:
    print("Set MALARIA_IMAGE_FILE to classify a single image (optional).")
else:
    image_path = Path(image_path_value).expanduser()
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
    image = tf.keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    image_array = tf.keras.utils.img_to_array(image) / 255.0
    score_for_index_1 = float(
        model.predict(image_array[None, ...], verbose=0)[0][0]
    )
    predicted_index = int(score_for_index_1 >= 0.5)
    index_to_class = {index: name for name, index in class_indices.items()}
    prediction = index_to_class[predicted_index]
    print(f"Predicted class: {prediction}")
